# LLMs As Re-Rankers

In [1]:
import pandas as pd


df = pd.read_json('llm-re-ranker-evaluations.jsonl', lines=True, orient='record')

In [2]:
df['re-ranking-family'] = df['llm_for_re_ranking'].apply(lambda i: i.split('-')[0])
df['eval-family'] = df['llm_for_eval'].apply(lambda i: i.split('-')[0])
df['same-family'] = df.apply(lambda i: i['re-ranking-family'] == i['eval-family'] and i["llm_for_re_ranking"] != i["llm_for_eval"], axis=1)
df['nDCG@10 Score (LLM - Original)'] = df['nDCG@10 Score (LLM)'] - df['nDCG@10 Score (original)']
df['nDCG@10 Rank (Original - LLM)'] = df['nDCG@10 Rank (original)'] - df['nDCG@10 Rank (LLM)']

In [3]:
GROUP_TO_RUN_TYPE = {
    # from https://arxiv.org/pdf/2003.07820
    'trec-dl-2019-judged': {
        'idst_bert_v3': 'nnlm', 'idst_bert_pr1': 'nnlm', 'idst_bert_r1': 'nnlm',
        'idst_bert_v2': 'nnlm', 'idst_bert_v1': 'nnlm', 'p_bert': 'nnlm',
        'idst_bert_r2': 'nnlm', 'bm25exp_marcomb': 'nnlm', 'TUW19-d3-re': 'nn',
        'ucas_runid1': 'nnlm', 'ucas_runid3': 'nnlm', 'runid3': 'nnlm',
        'bm25_marcomb': 'nnlm', 'bm25exp_marco': 'nnlm', 'ucas_runid2': 'nnlm',
        'runid2': 'nnlm', 'runid5': 'nnlm', 'TUW19-d2-re': 'nn',
        'uogTrDNN6LM': 'nnlm', 'TUW19-d1-re': 'nn', 'TUW19-p3-f': 'nn',
        'TUW19-p1-f': 'nn', 'TUW19-p3-re': 'nn', 'TUW19-p1-re': 'nn',
        'TUW19-p2-f': 'nn', 'ms_ensemble': 'nn', 'srchvrs_run1': 'trad',
        'TUW19-d2-f': 'nn', 'TUW19-d3-f': 'nn', 'TUW19-p2-f': 'nn',
        'dct_tp_bm25e2': 'nn', 'srchvrs_run2': 'trad', 'bm25tuned_rm3': 'trad',
        'dct_qp_bm25e': 'nn', 'dct_tp_bm25e': 'nn', 'uogTrDSSQE5LM': 'nnlm',
        'TUW19-d1-f': 'nn', 'ms_duet': 'nn', 'ms_duet_passage': 'nn',
        'uogTrDSS6pLM': 'nnlm', 'bm25tuned_prf': 'trad', 'bm25tuned_ax': 'trad',
        'bm25base': 'trad', 'bm25base_rm3': 'trad', 'runid1': 'nnlm',
        'bm25tuned': 'trad', 'bm25base_prf': 'trad', 'bm25base_prf_p': 'trad',
        'baseline': 'trad', 'bm25base_ax': 'trad', 'bm25tuned_rm3_p': 'trad',
        'bm25base_p': 'trad', 'ICT-BERT2': 'nnlm', 'ICT-CKNRM_B': 'nnlm',
        'ICT-CKNRM_B50': 'nnlm', 'p_exp_rm3_bert': 'nnlm', 'idst_bert_p1': 'nnlm',
        'idst_bert_p2': 'nnlm', 'idst_bert_p3': 'nnlm', 'idst_bert_pr2': 'nnlm',
        'idst_bert_pr1': 'nnlm', 'p_exp_bert': 'nnlm', 'test1': 'nnlm',
        'TUA1-1': 'nnlm', 'runid4': 'nnlm', 'srchvrs_ps_run2': 'nnlm',
        'TUW19-p2-re': 'nnlm', 'srchvrs_ps_run3': 'trad', 'bm25tuned_prf_p': 'trad',
        'bm25base_ax_p': 'trad', 'bm25tuned_ax_p': 'trad', 'bm25base_prf_p': 'trad',
        'bm25tuned_rm3_p': 'trad', 'bm25base_rm3_p': 'trad', 'bm25base_p': 'trad',
        'srchvrs_ps_run1': 'trad', 'bm25tuned_p': 'trad', 'UNH_bm25': 'trad',
        'UNH_exDL_bm25': 'trad',
    },
    # from https://trec.nist.gov/pubs/trec29/papers/OVERVIEW.DL.pdf
    'trec-dl-2020-judged': {
        'd_d2q_duo': 'nnlm', 'd_d2q_rm3_duo': 'nnlm', 'd_rm3_duo': 'nnlm',
        'ICIP_run1': 'nnlm', 'ICIP_run3': 'nnlm', 'fr_doc_roberta': 'nnlm',
        'ICIP_run2': 'nnlm', 'roberta-large': 'nnlm', 'bcai_bertb_docv': 'nnlm',
        'ndrm3-orc-full': 'nn', 'ndrm3-orc-re': 'nn', 'ndrm3-full': 'nn',
        'ndrm3-re': 'nn', 'ndrm1-re': 'nn', 'mpii_run2': 'nnlm',
        'bigIR-DTH-T5-R': 'nnlm', 'mpii_run1': 'nnlm', 'ndrm1-full': 'nn',
        'uob_runid3': 'nnlm', 'runid3': 'nnlm', 'bigIR-DTH-T5-F': 'nnlm',
        'd_d2q_bm25': 'nnlm', 'TUW-TKL-2k': 'nn', 'bigIR-DH-T5-R': 'nnlm',
        'uob_runid2': 'nnlm', 'uogTrQCBMP': 'nnlm', 'uob_runid1': 'nnlm',
        'TUW-TKL-4k': 'nn', 'bigIR-DH-T5-F': 'nnlm', 'bl_bcai_multfld': 'trad',
        'indri-sdmf': 'trad', 'bcai_classic': 'trad', 'longformer_1': 'nnlm',
        'uogTr31oR': 'nnlm', 'rterrier-expC2': 'trad', 'bigIR-DT-T5-R': 'nnlm',
        'uogTrT20': 'nnlm', 'RMIT_DFRee': 'trad', 'rmit_indri-fdm': 'trad',
        'd_d2q_bm25rm3': 'nnlm', 'rindri-bm25': 'trad', 'bigIR-DT-T5-F': 'nnlm',
        'bl_bcai_model1': 'trad', 'bl_bcai_prox': 'trad', 'terrier-jskls': 'trad',
        'rmit_indri-sdm': 'trad', 'rterrier-tfidf': 'trad', 'BIT-run2': 'nn',
        'RMIT_DPH': 'trad', 'd_bm25': 'trad', 'd_bm25rm3': 'trad',
        'rterrier-dph': 'trad', 'rterrier-tfidf2': 'trad', 'uogTrBaseQL17o': 'trad',
        'uogTrBaseL17o': 'trad', 'BIT-run1': 'nn', 'rterrier-dph_sd': 'trad',
        'BIT-run3': 'nn', 'uogTrBaseDPHQ': 'trad', 'uogTrBaseQL16': 'trad',
        'uogTrBaseL16': 'trad', 'uogTrBaseDPH': 'trad', 'nlm-bm25-prf-2': 'trad',
        'nlm-bm25-prf-1': 'trad', 'mpii_run3': 'nnlm', 'bm25tuned_rm3_p': 'trad',
        'pash_r3': 'nnlm', 'pash_r2': 'nnlm', 'pash_f3': 'nnlm',
        'pash_f1': 'nnlm', 'pash_f2': 'nnlm', 'p_d2q_bm25_duo': 'nnlm',
        'p_d2q_rm3_duo': 'nnlm', 'p_bm25rm3_duo': 'nnlm', 'CoRT-electra': 'nnlm',
        'RMIT-Bart': 'nnlm', 'pash_r1': 'nnlm', 'NLE_pr3': 'nnlm',
        'pinganNLP2': 'nnlm', 'pinganNLP3': 'nnlm', 'pinganNLP1': 'nnlm',
        'NLE_pr2': 'nnlm', 'NLE_pr1': 'nnlm', '1': 'nnlm',
        'bigIR-BERT-R': 'nnlm', 'fr_pass_roberta': 'nnlm', 'bigIR-DCT-T5-F': 'nnlm',
        'rr-pass-roberta': 'nnlm', 'bcai_bertl_pass': 'nnlm', 'bigIR-T5-R': 'nnlm',
        '2': 'nnlm', 'bigIR-T5-BERT-F': 'nnlm', 'bigIR-T5xp-T5-F': 'nnlm',
        'nlm-ens-bst-2': 'nnlm', 'nlm-ens-bst-3': 'nnlm', 'nlm-bert-rr': 'nnlm',
        'relemb_mlm_0_2': 'nnlm', 'nlm-prfun-bert': 'nnlm', 'TUW-TK-Sparse': 'nn',
        'TUW-TK-2Layer': 'nn', 'p_d2q_bm25': 'nnlm', 'p_d2q_bm25rm3': 'nnlm',
        'CoRT': 'nnlm', 'CoRT-bm25': 'nnlm', 'CoRT-standalone': 'nnlm',
        'DoRA_Large_1k': 'nnlm', 'DoRA_Small': 'nnlm', 'DoRA_Med': 'nnlm',
        'DoRA_Large': 'nnlm', 'med_1k': 'nnlm', 'bert_6': 'nnlm',
        'bcai_class_pass': 'trad', 'bl_bcai_mdl1_vt': 'trad', 'bl_bcai_mdl1_vs': 'trad',
        'indri-fdm': 'trad', 'terrier-InL2': 'trad', 'terrier-BM25': 'trad',
        'DLH_d_5_t_25': 'trad', 'indri-lmds': 'trad', 'indri-sdm': 'trad',
        'p_bm25rm3': 'trad', 'p_bm25': 'trad', 'terrier-DPH': 'trad',
        'bm25_bert_token': 'trad', 'TF_IDF_d_2_t_50': 'trad', 'small_1k': 'nnlm',
    },
}

df['type'] = df.apply(lambda i: GROUP_TO_RUN_TYPE[i['dataset']][i['run']], axis=1)

In [ ]:
def format_rank(number):
    ret = "{:.1f}".format(number)
    if len(ret) == 3:
        ret = '\\phantom{0}' + ret
    return ret

def format_score(number):
    return "{:.3f}".format(number).replace('0.', '.')


def table_row(allowed_systems, same_family):
    df_eval = df[df['same-family'] == same_family].copy()
    df_eval = df_eval[df_eval['type'].isin(allowed_systems)]
    ret = []
    
    for dataset in ['trec-dl-2019-judged', 'trec-dl-2020-judged']:
        d = df_eval[df_eval['dataset'] == dataset]
        score_data = d['nDCG@10 Rank (Original - LLM)'].describe().to_dict()
        ret += [
            format_rank(score_data['25%']),
            format_rank(score_data['mean']),
            format_rank(score_data['75%']),
        ]
        

        score_data = d['nDCG@10 Score (LLM - Original)'].describe().to_dict()
        ret += [
            format_score(score_data['25%']),
            format_score(score_data['mean']),
            format_score(score_data['75%']),
        ]
    
    return ' & '.join(ret)


def table():
    return """\\begin{tabular}{@{}ll@{\\hspace{.4em}}ccc@{\\hspace{1em}}ccc@{\\hspace{1em}}ccc@{\\hspace{1em}}ccc@{}}
\\toprule
\\multicolumn{2}{@{}l@{}}{\\bfseries Systems}      &   \\multicolumn{6}{c@{\\hspace{1em}}}{\\bfseries DL 19}  & \\multicolumn{6}{c@{\\hspace{1em}}}{\\bfseries DL 20}\\\\

\\cmidrule(r@{.5em}){3-8}
\\cmidrule{9-14}

& &  \\multicolumn{3}{c@{\\hspace{1em}}}{$\\Delta$ Rank} &   \\multicolumn{3}{c@{\\hspace{1em}}}{$\\Delta$ Score} &   \\multicolumn{3}{c@{\\hspace{1em}}}{$\\Delta$ Rank} &   \\multicolumn{3}{c@{\\hspace{1em}}}{$\\Delta$ Score} \\\\

\\cmidrule(r@{.5em}){3-5}
\\cmidrule(r@{.5em}){6-8}
\\cmidrule(r@{.5em}){9-11}
\\cmidrule{12-14}


& & 25\\,\\% & Avg.\\ & 75\\,\\% & 25\\,\\% & Avg.\\ & 75\\,\\% & 25\\,\\% & Avg.\\ & 75\\,\\% & 25\\,\\% & Avg.\\ & 75\\,\\% \\\\

\\midrule

\\parbox[t]{2mm}{\\multirow{4}{*}{\\rotatebox[origin=c]{90}{Within}}} & NNLM & """ + table_row(['nnlm'], True) + """ \\\\

& NN & """ + table_row(['nn'], True) + """ \\\\

& Trad & """ + table_row(['trad'], True) + """ \\\\

\\cmidrule{2-14}

& All & """ + table_row(['nn', 'nnlm', 'trad'], True) + """ \\\\

\\midrule



\\parbox[t]{2mm}{\\multirow{4}{*}{\\rotatebox[origin=c]{90}{Accross}}} & NNLM & """ + table_row(['nnlm'], False) + """ \\\\

& NN & """ + table_row(['nn'], False) + """ \\\\

& Trad & """ + table_row(['trad'], False) + """ \\\\

\\cmidrule{2-14}

& All & """ + table_row(['nn', 'nnlm', 'trad'], False) + """ \\\\

\\bottomrule

\\end{tabular}"""

In [5]:
print(table())

\begin{tabular}{@{}ll@{\hspace{.4em}}ccc@{\hspace{1em}}ccc@{\hspace{1em}}ccc@{\hspace{1em}}ccc@{}}
\toprule
\multicolumn{2}{@{}l@{}}{\bfseries Systems}      &   \multicolumn{6}{c@{\hspace{1em}}}{\bfseries DL 19}  & \multicolumn{6}{c@{\hspace{1em}}}{\bfseries DL 20}\\

\cmidrule(r@{.5em}){4-9}
\cmidrule{10-14}

& &  \multicolumn{3}{c@{\hspace{1em}}}{$\Delta$ Rank} &   \multicolumn{3}{c@{\hspace{1em}}}{$\Delta$ Score} &   \multicolumn{3}{c@{\hspace{1em}}}{$\Delta$ Rank} &   \multicolumn{3}{c@{\hspace{1em}}}{$\Delta$ Score} \\

\cmidrule(r@{.5em}){3-5}
\cmidrule(r@{.5em}){6-8}
\cmidrule(r@{.5em}){9-11}
\cmidrule{12-14}


& & 25\,\% & Avg.\ & 75\,\% & 25\,\% & Avg.\ & 75\,\% & 25\,\% & Avg.\ & 75\,\% & 25\,\% & Avg.\ & 75\,\% \\

\midrule

\parbox[t]{2mm}{\multirow{4}{*}{\rotatebox[origin=c]{90}{Within}}} & NNLM & \phantom{0}9.0 & 10.5 & 13.0 & .146 & .176 & .200 & \phantom{0}9.0 & 17.5 & 27.0 & .166 & .200 & .244 \\

& NN & \phantom{0}9.8 & 11.2 & 13.2 & .147 & .181 & .202 & 10.2 & 17.5 &

## Pre-Calculate stuff

In [ ]:
from glob import glob
from trectools import TrecRun, TrecQrel, TrecEval
from tqdm import tqdm
import pandas as pd
import json

DATASET_TO_TREC_IDENTIFIER = {
    'trec-dl-2019-judged': 'trec28',
    'trec-dl-2020-judged': 'trec29',
}

MODEL_TO_NAME = {
    'AnthropicLLM-claude-3-haiku-20240307-umbrella_zeroshot_basic': 'Claude-3-haiku',
    'AnthropicLLM-claude-3-sonnet-20240229-umbrella_zeroshot_basic': 'Claude-3-sonnet',
    'LiteLLM-llama3-umbrella_zeroshot_basic': 'Llama-3',
    'LiteLLM-llama3.1-umbrella_zeroshot_basic': 'Llama-3.1',
    'GeminiGPT-gemini-1.5-flash-umbrella_zeroshot_basic': 'Gemini-1.5-flash',
    'GeminiGPT-gemini-1.5-flash-8b-umbrella_zeroshot_basic': 'Gemini-1.5-flash-8b',
    'OpenAiGPT-gpt-4o-umbrella_zeroshot_basic': 'GPT-4o',
    'OpenAiGPT-gpt-4o-mini-umbrella_zeroshot_basic': 'GPT-4o-mini',
}

RUNS = {}
QRELS = {}

def load_qrels(dataset, name):
    path = f'../data/msmarco-passage-{dataset}/qrels/*.qrels.txt'

    global qrels
    if dataset not in QRELS:
        QRELS[dataset] = {}
        for i in tqdm(glob(path), 'load qrels'):
            qrel_name = i.split('/')[-1].split('.qrels')[0]
            assert qrel_name and qrel_name not in QRELS[dataset]
            QRELS[dataset][qrel_name] = TrecQrel(i)
    
    ret = TrecQrel()
    ret.qrels_data = QRELS[dataset][name].qrels_data.copy()
    return ret

def load_runs(dataset):
    ret = {}

    path = f'../data/trec-system-runs/{DATASET_TO_TREC_IDENTIFIER[dataset]}/deep.passages/input.*.gz'

    global RUNS

    if dataset not in RUNS:
        topics = load_qrels(dataset, 'trec').topics()
        for i in tqdm(glob(path), 'load runs'):
            run_name = i.split('/')[-1].split('.')[1]
            assert run_name and run_name not in ret
            ret[run_name] = TrecRun(i)
            ret[run_name].run_data = ret[run_name].run_data[ret[run_name].run_data['query'].isin(topics)]
            
        RUNS[dataset] = ret
        ret = {}

    for run_name, run in RUNS[dataset].items():
        run_copy = TrecRun()
        run_copy.run_data = run.run_data.copy()
        ret[run_name] = run_copy

    return ret

def re_rank_with_llm(run, dataset, llm_for_re_ranking):
    qrels = load_qrels(dataset, llm_for_re_ranking)
    scores = {}
    for _, i in qrels.qrels_data.iterrows():
        if str(i['query']) not in scores:
            scores[str(i['query'])] = {}
        scores[str(i['query'])][str(i['docid'])] = int(i['rel'])

    ret = TrecRun()
    ret.run_data = run.run_data.copy()
    ret.run_data
    ret.run_data['score'] = ret.run_data.apply(lambda i: scores[str(i['query'])].get(str(i['docid']), -1), axis=1)

    trecformat = ret.run_data.sort_values(["query", "score", "docid"], ascending=[True,False,False]).reset_index()
    topX = trecformat.groupby("query")[["query","docid","score"]].head(1000)
    topX["rank"] = 1
    topX["rank"] = topX.groupby("query")["rank"].cumsum()
    ret.run_data = topX

    return ret

def evaluate(dataset, run_to_modify, llm_for_re_ranking, llm_for_eval):
    qrels = {
        'original': load_qrels(dataset, 'trec'),
        'llm_eval': load_qrels(dataset, llm_for_eval),
    }
    evals = []

    for run_name, run in load_runs(dataset).items():
        evals.append({'run': run_name, 'original': TrecEval(run=run, qrels=qrels['original']).get_ndcg(depth=10), 'llm': TrecEval(run=run, qrels=qrels['llm_eval']).get_ndcg(depth=10)})

        if run_name == run_to_modify:
            modified_run = re_rank_with_llm(run, dataset, llm_for_re_ranking)
            evals.append({'run': run_name + '-re-ranked', 'original': TrecEval(run=modified_run, qrels=qrels['original']).get_ndcg(depth=10), 'llm': TrecEval(run=modified_run, qrels=qrels['llm_eval']).get_ndcg(depth=10)})


    evals = pd.DataFrame(evals)

    evals = evals.sort_values(["original"], ascending=[False]).reset_index()
    evals["rank_original"] = 1
    evals["rank_original"] = evals["rank_original"].cumsum()

    evals = evals.sort_values(["llm"], ascending=[False]).reset_index()
    evals["llm_rank"] = 1
    evals["llm_rank"] = evals["llm_rank"].cumsum()

    re_rank_eval = evals[evals['run'] == run_to_modify + '-re-ranked']
    assert len(re_rank_eval) == 1
    re_rank_eval = re_rank_eval.iloc[0].to_dict()

    return {
        'run': run_to_modify,
        'dataset': dataset,
        'llm_for_re_ranking': llm_for_re_ranking,
        'llm_for_eval': llm_for_eval,
        'nDCG@10 Score (original)': re_rank_eval['original'],
        'nDCG@10 Rank (original)': re_rank_eval['rank_original'],
        'nDCG@10 Score (LLM)': re_rank_eval['llm'],
        'nDCG@10 Rank (LLM)': re_rank_eval['llm_rank'],
    }


In [2]:
for dataset in DATASET_TO_TREC_IDENTIFIER:
    load_runs(dataset)
    load_qrels(dataset, 'trec')


load runs: 100%|██████████| 59/59 [00:35<00:00,  1.67it/s]


In [7]:
def all_test_permutations():
    for dataset in DATASET_TO_TREC_IDENTIFIER:
        for run in load_runs(dataset):
            for evaluation_model in MODEL_TO_NAME:
                for re_rank_model in MODEL_TO_NAME:
                    yield (dataset, run, evaluation_model, re_rank_model)

with open('llm-re-ranker-evaluations.jsonl', 'w') as f:
    for dataset, run, evaluation_model, re_rank_model in tqdm(list(all_test_permutations()), 'Create Re-Rank Evaluations'):
        f.write(json.dumps(evaluate(dataset, run, evaluation_model, re_rank_model)) + '\n')
        f.flush()

Create Re-Rank Evaluations: 100%|██████████| 6144/6144 [4:01:38<00:00,  2.36s/it]  
